In [1]:
import pandas as pd
from pathlib import Path

# === 1.加载 species 列表 ===
targets = pd.read_excel("output/3_grid_search/pathogen_best_params.xlsx")["species"]

# === 2. 文件目录 ===
base_dir = Path("output/4_predict/pathogen/rc")
df_list = []

for sp in targets:
    df = (
        pd.read_excel(base_dir / f"{sp}.xlsx", usecols=["Key", "std_sum"])
          .assign(Key=lambda d: d["Key"].astype(str).str.strip())
          .set_index("Key")
          .rename(columns={"std_sum": sp})
    )
    df_list.append(df)

# === 3. 高效按列合并（自动对齐 Key） ===
merged_df = pd.concat(df_list, axis=1, join="outer").round(3)

# === 4. 输出 ===
out_path = Path("output/5_calculate/pathogen/rc.xlsx")
out_path.parent.mkdir(parents=True, exist_ok=True)
merged_df.to_excel(out_path)

print(f"✔ 合并完成，共 {len(df_list)} 个 host 文件 → {out_path}")

✔ 合并完成，共 8 个 host 文件 → output\5_calculate\pathogen\rc.xlsx


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# === 1. 读取 species 列表 ===
targets = pd.read_excel("output/3_grid_search/pathogen_best_params.xlsx")["species"]

input_dir = Path("output/4_predict/pathogen/prob")

# === 2. 计算并保存统计值 ===
def calculate_probability(species):
    df = pd.read_excel(input_dir / f"{species}.xlsx")
    # print(species, "min threshold=", df['threshold'].min(), 'max threshold=', df['threshold'].max())

    grouped = (
        df.groupby("gb")["pred"]
          .agg(
              mean="mean",
              lbd=lambda x: np.quantile(x, 0.025),
              ubd=lambda x: np.quantile(x, 0.975),
          )
          .round(3)
          .reset_index()
    )

    out_path = input_dir / f"{species}_statistic.xlsx"
    grouped.to_excel(out_path, index=False)

    print(f"✔ {species} 完成统计 → {out_path}")

# === 3. 批量执行 ===
for species in targets:
    calculate_probability(species)

✔ Hantaan virus 完成统计 → output\4_predict\pathogen\prob\Hantaan virus_statistic.xlsx
✔ Seoul virus 完成统计 → output\4_predict\pathogen\prob\Seoul virus_statistic.xlsx
✔ Wenzhou arenavirus 完成统计 → output\4_predict\pathogen\prob\Wenzhou arenavirus_statistic.xlsx
✔ Encephalomyocarditis virus 完成统计 → output\4_predict\pathogen\prob\Encephalomyocarditis virus_statistic.xlsx
✔ Rat hepatitis E virus 完成统计 → output\4_predict\pathogen\prob\Rat hepatitis E virus_statistic.xlsx
✔ Betacoronavirus 完成统计 → output\4_predict\pathogen\prob\Betacoronavirus_statistic.xlsx
✔ Morbillivirus 完成统计 → output\4_predict\pathogen\prob\Morbillivirus_statistic.xlsx
✔ Henipavirus 完成统计 → output\4_predict\pathogen\prob\Henipavirus_statistic.xlsx


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

def calculate_high_risk_county():

    # ==== 1. 读取静态数据（一次即可） ====
    county_info = pd.read_excel("input/China_county_level_population_area.xlsx")
    data = pd.read_excel("../1_clean/data_merge_all_20260110.xlsx")
    data = data[~(data['host_species'].isna() & data['standard virus name'].isna())].reset_index(drop=True) #key
    print(data.shape[0])
    
    # === 2. 生成 case/control 基础表 ===
    case_control_df = (
        data.assign(presence=1)
        .pivot_table(
            index="gb",
            columns="standard virus name",
            values="presence",
            aggfunc="max",
            fill_value=0
        )
        .reset_index()
    )

    # ==== 3. 读取 species 列表 ====
    targets = pd.read_excel("output/3_grid_search/pathogen_best_params.xlsx")["species"]

    prob_dir = Path("output/4_predict/pathogen/prob")

    # ==== 4. 工具函数 ====
    def compute_stats(gb_list):
        subset = county_info[county_info["gb"].isin(gb_list)]
        return (
            len(subset),
            subset["area(km2)"].sum(),
            subset["population(1e4)"].sum(),
        )

    def pct_change(pred, obv):
        return np.nan if obv == 0 else 100 * (pred - obv) / obv

    # ==== 5. 主循环 ====
    results = []

    for species in targets:
        # ----- A. 观测数据 -----
        observed_gb = case_control_df.loc[case_control_df[species] == 1, "gb"]

        obv_num, obv_area, obv_pop = compute_stats(observed_gb)
        obv_area_m = obv_area / 1e6    # 百万 km²
        obv_pop_m = obv_pop / 1e2      # 百万

        # ----- B. 预测数据 -----
        pred_df = pd.read_excel(prob_dir / f"{species}_statistic.xlsx")
        pred_gb = pred_df.loc[pred_df["mean"] >= 0.5, "gb"]

        pred_num, pred_area, pred_pop = compute_stats(pred_gb)
        pred_area_m = pred_area / 1e6
        pred_pop_m = pred_pop / 1e2

        # ----- C. 格式化 -----
        county_str = f"{pred_num}/{obv_num} ({pct_change(pred_num, obv_num):.1f})"
        area_str   = f"{pred_area_m:.2f}/{obv_area_m:.2f} ({pct_change(pred_area_m, obv_area_m):.1f})"
        pop_str    = f"{pred_pop_m:.2f}/{obv_pop_m:.2f} ({pct_change(pred_pop_m, obv_pop_m):.1f})"

        results.append([species, county_str, area_str, pop_str])

    # ==== 6. 输出表 ====
    df_out = pd.DataFrame(results, columns=["pathogen", "county", "area", "population"])

    outdir = Path("output/5_calculate/pathogen")
    outdir.mkdir(parents=True, exist_ok=True)

    out_path = outdir / "pc.xlsx"
    df_out.to_excel(out_path, index=False)

    print(f"✔ 完成：共计算 {len(df_out)} 个 pathogen  → {out_path}")


# ==== 执行 ====
calculate_high_risk_county()

47196
✔ 完成：共计算 8 个 pathogen  → output\5_calculate\pathogen\pc.xlsx


In [4]:
import pandas as pd
from pathlib import Path

# ============================================================
# 读取指标文件（mean / lbd / ubd）
# ============================================================
def read_metric_value(filepath):
    df = pd.read_excel(filepath)
    df = df.set_index(df['index'].str.lower())
    return (
        float(df.loc["mean", "value"]),
        float(df.loc["lbd", "value"]),
        float(df.loc["ubd", "value"])
    )

# ============================================================
# 主流程
# ============================================================
def calculation_table():
    targets = pd.read_excel("output/3_grid_search/pathogen_best_params.xlsx")["species"]

    metrics = ["auc", "tpr", "tnr"]
    base_dir = Path("output/4_predict/pathogen")

    results = []

    for species in targets:
        row = {"pathogen": species}

        for metric in metrics:
            f = base_dir / metric / f"{species}.xlsx"
            mean, lbd, ubd = read_metric_value(f)
            row[metric] = f"{mean:.3f} ({lbd:.3f}, {ubd:.3f})"

        results.append(row)

    result_df = pd.DataFrame(results)

    outdir = Path("output/5_calculate/pathogen")
    outdir.mkdir(parents=True, exist_ok=True)

    output_file = outdir / "metrics.xlsx"
    result_df.to_excel(output_file, index=False)
    print(f"✔ 保存完成 → {output_file}")

    return result_df

# 运行
df_result = calculation_table()

✔ 保存完成 → output\5_calculate\pathogen\metrics.xlsx
